<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/Simulated_Annealing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving knapsack-20.txt to knapsack-20.txt
Saving knapsack-200.txt to knapsack-200.txt


In [ ]:
def read_tsp_file(filename):
    locations = []
    reading_coords = False
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                reading_coords = True
                continue
            if line == "EOF" or not line: break
            if reading_coords:
                parts = line.split()
                locations.append((float(parts[1]), float(parts[2])))
    return locations

In [ ]:
import math
import random
def get_dist(c1, c2):
    return math.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)

In [ ]:
def calculate_total_distance(route, locations):
    d = 0
    for i in range(len(route)):
        d += get_dist(locations[route[i-1]], locations[route[i]])
    return d

In [ ]:
def simulated_annealing_tsp(locations, initial_temp, cooling_rate, t_min, max_iter):
    n = len(locations)
    current_sol = list(range(n))
    random.shuffle(current_sol)
    current_cost = calculate_total_distance(current_sol, locations)

    best_sol = list(current_sol)
    best_cost = current_cost
    temp = initial_temp
    total_steps = 0


    while temp > t_min:


        for _ in range(max_iter):
            total_steps += 1

            idx1, idx2 = random.sample(range(n), 2)


            def get_local_dist(sol, i):
                return get_dist(locations[sol[i-1]], locations[sol[i]]) + \
                       get_dist(locations[sol[i]], locations[sol[(i+1)%n]])

            old_dist = get_local_dist(current_sol, idx1) + get_local_dist(current_sol, idx2)
            current_sol[idx1], current_sol[idx2] = current_sol[idx2], current_sol[idx1]
            new_dist = get_local_dist(current_sol, idx1) + get_local_dist(current_sol, idx2)

            cost_diff = new_dist - old_dist

            accepted = False
            if cost_diff < 0:
                accepted = True
            else:
                prob = math.exp(-cost_diff / temp)
                if random.random() < prob:
                    accepted = True

            if accepted:
                current_cost += cost_diff
                if current_cost < best_cost:
                    best_cost = current_cost
                    best_sol = list(current_sol)
            else:

                current_sol[idx1], current_sol[idx2] = current_sol[idx2], current_sol[idx1]


        temp *= cooling_rate

    return best_cost, total_steps

In [ ]:
import time
import csv
instances = ["kroC100.tsp", "fi10639.tsp"]
configs = [
    {"init_t": 10000, "alpha": 0.99, "t_min": 0.001, "max_iter": 100},   # Fast
    {"init_t": 10000, "alpha": 0.999, "t_min": 0.001, "max_iter": 1000}   # Slow
]

all_results = []

for file_name in instances:
    locs = read_tsp_file(file_name)
    for cfg in configs:
        print(f"Testing {file_name} with alpha={cfg['alpha']}...")
        start_time = time.time()

        # Capture both the cost and the iterations from the new function
        best_cost, total_iters = simulated_annealing_tsp(
            locs, cfg['init_t'], cfg['alpha'], cfg['t_min'], cfg['max_iter']
        )

        elapsed = time.time() - start_time

        all_results.append({
            "Instance": file_name,
            "Alpha": cfg['alpha'],
            "T_min": cfg['t_min'],
            "Iterations": total_iters,
            "Best_Cost": round(best_cost, 2),
            "Time_Sec": round(elapsed, 2)
        })


with open('sa_final_experiments.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
    writer.writeheader()
    writer.writerows(all_results)

print("Experiments complete. Data saved to sa_final_experiments.csv")

Testing kroC100.tsp with alpha=0.99...
Testing kroC100.tsp with alpha=0.999...
Testing fi10639.tsp with alpha=0.99...
Testing fi10639.tsp with alpha=0.999...
Experiments complete. Data saved to sa_final_experiments.csv


**Problem Definition:**

The Knapsack Problem involves selecting a subset of items to maximize total value while ensuring the sum of their weights does not exceed a maximum capacity $W$.

In [ ]:
import random
import math
import time
import csv

def read_knapsack_instance(filename):
  items = []
  try:
    with open(filename, 'r') as f:
      lines = f.readlines()
      capacity = float(lines[-1].strip())
      for line in lines[1:-1]:
        parts = line.split()
        if len(parts) >= 3:
          items.append((float(parts[1]), float(parts[2])))
    return items, capacity
  except Exception as e:
    print(f"Error reading {filename}: {e}")
    return [], 0


In [ ]:
def evaluate(solution, items, capacity):
  total_value = 0
  total_weight = 0
  for i in range(len(solution)):
    if solution[i] == 1:
      total_value += items[i][0]
      total_weight += items[i][1]

  if total_weight > capacity:
    return 0
  return total_value

Algorithm: **Simulated Annealing** (Maximization):

Initial State: A random binary vector.

Neighborhood: Flipping a single bit (adding or removing an item).

Acceptance: Better moves are always accepted. Worse moves are accepted with a probability $e^{\Delta V/T}$ to allow the algorithm to escape local maxima.

Cooling: The temperature $T$ is reduced geometrically ($T = \alpha T$).

**Parameter settings**



**Initial Temp (T):**

High values (e.g., 1000+) ensure the initial random solution can be completely reshuffled to escape poor starting states.

**Cooling Rate (α):**

Values closer to 1 (e.g., 0.99) allow for a ""Slow Anneal,"" which is superior for the 200-item problem as it prevents the algorithm from getting stuck in local maxima too early.

**Inner Iterations:**

Spending more time at each temperature level (the inner repeat loop) helps the system reach thermal equilibrium, leading to more stable and higher-quality results.

In [ ]:
def simulated_annealing_knapsack(items, capacity, T_init, alpha, T_min, max_inner_iter):
  n = len(items)
  current_sol = [random.randint(0,1) for _ in range(n)]
  current_value = evaluate(current_sol, items, capacity)

  best_sol = list(current_sol)
  best_value = current_value

  T = T_init
  while T > T_min:
    for _ in range(max_inner_iter):
      idx = random.randint(0,n-1)
      current_sol[idx] = 1 - current_sol[idx]
      neighbor_val = evaluate(current_sol,items,capacity)

      if neighbor_val > current_value:
        current_value = neighbor_val
      else:
        delta = neighbor_val - current_value
        if random.random() <  math.exp(delta/T):
          current_value = neighbor_val
        else:
          current_sol[idx] = 1 - current_sol[idx]
      if current_value > best_value:
        best_value = current_value
        best_sol = list(current_sol)
    T *= alpha

  return best_value




In [ ]:
instances = ["knapsack-20.txt", "knapsack-200.txt"]
configs = [
    {"name": "Fast_Quench", "T_init": 1000, "alpha": 0.85, "T_min": 0.1, "max_inner": 50},
    {"name": "Slow_Anneal", "T_init": 1000, "alpha": 0.99, "T_min": 0.1, "max_inner": 150}
]
output_data = []
num_runs = 5
for file_name in instances:
  items, capacity = read_knapsack_instance(file_name)
  if not items: continue
  for cfg in configs:
        print(f"Running {file_name} | Config: {cfg['name']}...")
        run_values = []
        start_time = time.time()

        for _ in range(num_runs):
            val = simulated_annealing_knapsack(
                items, capacity, cfg['T_init'], cfg['alpha'], cfg['T_min'], cfg['max_inner']
            )
            run_values.append(val)

        avg_time = (time.time() - start_time) / num_runs

        output_data.append({
            "Instance": file_name,
            "Config": cfg['name'],
            "T_init": cfg['T_init'],
            "Alpha": cfg['alpha'],
            "Inner_Iter": cfg['max_inner'],
            "Best_Val": max(run_values),
            "Avg_Val": sum(run_values) / num_runs,
            "Avg_Time_Sec": round(avg_time, 4),
            "Num_Runs": num_runs
        })

with open('A4_knapsack_results.csv', 'w', newline='') as f:
  writer = csv.DictWriter(f, fieldnames = output_data[0].keys())
  writer.writeheader()
  writer.writerows(output_data)

print("Done. Experiments saved in the csv file")

Running knapsack-20.txt | Config: Fast_Quench...
Running knapsack-20.txt | Config: Slow_Anneal...
Running knapsack-200.txt | Config: Fast_Quench...
Running knapsack-200.txt | Config: Slow_Anneal...
Done. Experiments saved in the csv file


**Comparative results of experiments:**

|Instance|Config|T\_init|Alpha|Inner\_Iter|Best\_Val|Avg\_Val|Avg\_Time\_Sec|Num\_Runs|
|---|---|---|---|---|---|---|---|---|
|knapsack-20\.txt|Fast\_Quench|1000|0\.85|50|726\.0|695\.8|0\.0074|5|
|knapsack-20\.txt|Slow\_Anneal|1000|0\.99|150|726\.0|726\.0|0\.3617|5|
|knapsack-200\.txt|Fast\_Quench|1000|0\.85|50|132409\.0|131978\.2|0\.0413|5|
|knapsack-200\.txt|Slow\_Anneal|1000|0\.99|150|134543\.0|133677\.0|2\.2943|5|

**Discussion of results**

**Comparison of Instances**

**knapsack-20.txt**: For the small instance, both configurations achieved a Best Value of 726.0. However, the Slow_Anneal configuration was more stable, reaching an Average Value of 726.0 across all 5 runs, while the Fast_Quench varied slightly with an average of 695.8. This suggests that for small problem spaces, rapid cooling is sufficient to find the optimum, but slower cooling guarantees consistency.

**knapsack-200.txt**: In the larger search space, the impact of the cooling schedule is more pronounced. The Slow_Anneal configuration outperformed Fast_Quench by reaching a Best Value of 134,543.0, compared to 132,409.0.

**Parameter Impact Analysis**

**Cooling Rate ($\alpha$)**: The transition from $\alpha = 0.85$ to $\alpha = 0.99$ provided the algorithm with more time to explore the combinatorial space. In the 200-item instance, this slower cooling allowed the algorithm to escape local maxima, resulting in a 1.6% increase in the best solution found.

**Inner Iterations (max_inner)**: Spending 150 iterations at each temperature level (compared to 50) helped the system reach thermal equilibrium. This is evident in the 200-item instance, where the average solution quality increased from 131,978.2 to 133,677.0.

**Time vs. Quality Trade-off**: While Slow_Anneal provided superior results, it required significantly more computational time (approximately 2.29 seconds per run for the large instance compared to 0.04 seconds for the fast version). For this problem scale, the extra 2 seconds is a negligible price to pay for a better optimization result.